In [ ]:
import pandas as pd
import numpy as np
import json

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
style = "steampunk"
model = "FLUX.1-schnell"

In [ ]:
clip_data = pd.read_csv(f'results_{model}_mean_ot_{style}_42/calculate_clip_score/clip_score.csv', index_col=0)

clip_data.head()


In [ ]:
clip_data_pid = pd.read_csv(f'results_{model}_mean_ot_pid_{style}_42/calculate_clip_score/clip_score.csv', index_col=0)

clip_data.head()


In [ ]:
fig = go.Figure()

lbs = clip_data['strength'].unique()

zeroshot_mean_ot = [clip_data[
    (clip_data['strength'] == lb) & (clip_data['conditional_zero_shot_score'] >= 0.49)
].shape[0] / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs]

zeroshot_mean_ot_pid = [clip_data_pid[
    (clip_data_pid['strength'] == lb) & (clip_data_pid['conditional_zero_shot_score'] >= 0.49)
].shape[0] / clip_data_pid[clip_data_pid['strength'] == 1].shape[0] for lb in lbs]

# --- First plot: line across all points ---
fig.add_trace(go.Scatter(
    x=lbs,
    y=zeroshot_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT"
))

# --- Clip Data (PID) ---
fig.add_trace(go.Scatter(
    x=lbs,
    y=zeroshot_mean_ot_pid,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="Mean-AcT-PID"
))

# Layout
fig.update_layout(
    paper_bgcolor="white",
    width=700,
    height=500,
    showlegend=False,
    legend=dict(font=dict(size=18)),  # legend font
    font=dict(size=18),               # default font for all text
    xaxis=dict(
        domain=[0, 1.0],
        title="Strength",
        tickfont=dict(size=16)
    ),
    yaxis=dict(
        title="0-shot(style) (→)",
        # titlefont=dict(size=20),
        tickfont=dict(size=16)
    ),
    margin=dict(t=40, b=60, l=80, r=20)
)

fig.show()
fig.write_image(f"visualization/{model}/{style}/0shot_clip_{style}.pdf", width=700, height=500, scale=2)

In [ ]:
fig = go.Figure()

lbs = clip_data['strength'].unique()

clipscore_mean_ot = [clip_data['conditional_similarity'][
    (clip_data['strength'] == lb) & (clip_data['unconditional_similarity'] >= 0.0)
].sum() / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs]

clipscore_mean_ot_pid = [clip_data_pid['conditional_similarity'][
    (clip_data_pid['strength'] == lb) & (clip_data_pid['unconditional_similarity'] >= 0.0)
].sum()/ clip_data_pid[clip_data_pid['strength'] == 1].shape[0] for lb in lbs]

# --- First plot: line across all points ---
fig.add_trace(go.Scatter(
    x=lbs,
    y=clipscore_mean_ot,
    mode="lines+markers",
    line=dict(color='#4C9AFF', width=2),
    marker=dict(size=8),
    name="Mean-AcT"
))

# --- Clip Data (PID) ---
fig.add_trace(go.Scatter(
    x=lbs,
    y=clipscore_mean_ot_pid,
    mode="lines+markers",
    line=dict(color='#FFB347', width=2),
    marker=dict(size=8),
    name="PID-AcT"
))

# Layout
fig.update_layout(
    width=700,
    height=500,
    showlegend=True,
    xaxis=dict(domain=[0, 1.0], title="Strength"),
    yaxis=dict(title="ClipScores (style) (→)"),
    margin=dict(t=20, b=60)
)

fig.show()
fig.write_image(f"visualization/{model}/{style}/clipscore_{style}.pdf", width=700, height=500, scale=2)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplots: 2 rows, 1 column (stacked)
fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=True,
    horizontal_spacing=0.1  # space between the two panels
)

# (Optional) ensure ordered x-axis
lbs = sorted(clip_data['strength'].unique())

# --- Zeroshot values ---
zeroshot_mean_ot = [
    clip_data[
        (clip_data['strength'] == lb) & (clip_data['conditional_zero_shot_score'] >= 0.49)
    ].shape[0] / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs
]
zeroshot_mean_ot_pid = [
    clip_data_pid[
        (clip_data_pid['strength'] == lb) & (clip_data_pid['conditional_zero_shot_score'] >= 0.49)
    ].shape[0] / clip_data_pid[clip_data_pid['strength'] == 1].shape[0] for lb in lbs
]

# --- CLIPScore values ---
clipscore_mean_ot = [
    clip_data['unconditional_similarity'][
        (clip_data['strength'] == lb) & (clip_data['unconditional_similarity'] >= 0.0)
    ].sum() / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs
]
clipscore_mean_ot_pid = [
    clip_data_pid['unconditional_similarity'][
        (clip_data_pid['strength'] == lb) & (clip_data_pid['unconditional_similarity'] >= 0.0)
    ].sum() / clip_data_pid[clip_data_pid['strength'] == 1].shape[0] for lb in lbs
]

# --- Top subplot (row 1): Zeroshot ---
fig.add_trace(go.Scatter(
    x=lbs, y=zeroshot_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=lbs, y=zeroshot_mean_ot_pid,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="PID-AcT"  # fix label
), row=1, col=1)

# --- Bottom subplot (row 2): CLIPScore ---
fig.add_trace(go.Scatter(
    x=lbs, y=clipscore_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT",
    showlegend=False  # avoid duplicate legends
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=lbs, y=clipscore_mean_ot_pid,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="PID-AcT",
    showlegend=False
), row=1, col=2)

# Layout: put legend below, x-title only on bottom subplot
fig.update_layout(
    width=900,
    height=250,             # taller for vertical stack
    showlegend=True,
    legend=dict(
        orientation="v",     # keep horizontal
        yanchor="top",       # anchor to top
        y=0.28,              # a bit below the top edge of fig
        xanchor="right",     # anchor relative to right
        x=0.80,              # place near the right edge
        bgcolor="rgba(255,255,255,0.6)",  # optional: semi-transparent background
        bordercolor="black",  # optional: border
        borderwidth=1,
        font=dict(size=15)
    ),
    margin=dict(t=10, b=40, r=40),
)

# Axis titles
fig.update_yaxes(title_text="0-shot (→)", row=1, col=1)
fig.update_yaxes(title_text="CLIPScore (→)", row=2, col=1)
fig.update_xaxes(title_text="Strength", row=2, col=1)  # bottom only

# Fonts / ticks
fig.update_layout(font=dict(size=30))
fig.update_xaxes(tickfont=dict(size=25), title_font=dict(size=30), tickformat=".2f")
fig.update_yaxes(tickfont=dict(size=25), title_font=dict(size=30), tickformat=".2f")

fig.show()

# Save (increase height for PDF export too)
fig.write_image(
    f"visualization/{model}/{style}/combined_{style}.pdf",
    width=1000, height=250, scale=2
)


In [ ]:
# data_dir = "results_42"

In [ ]:
# zero_shot_res = pd.read_csv(data_dir + '/' + 'evaluate_0shot/0shot_eval.csv', index_col=0)
# zero_shot_res[zero_shot_res['q0_llm_answer'] == "Yes"].shape[0] / zero_shot_res.shape[0]

In [ ]:
# mmlu_res = pd.read_pickle(data_dir+'/evaluate_eleuther/eleuther.pkl')

In [ ]:
# mmlu_res['results']['mmlu']['acc,none']

In [ ]:
# mistral_res = pd.read_csv(data_dir+'/evaluate_perplexity/model_perplexity.csv', index_col=0)
# mistral_res[mistral_res['strength']==1.0]['ppl_Mistral-7B-v0.1'].mean()